In [9]:
import os
import json
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kurtosis

# =========================================================
# 1) 入力ロード
# =========================================================
def search_and_load_spike_data(
    base_dir, mouse_id, experience_level, trial_type, specific_image, image_set, max_length=75
):
    """
    base_dir/{mouse_id}/{experience_level}/{image_set}/
        spike_data_{mouse_id}_{experience_level}_{image_set}_{trial_type}_{specific_image}.npy
    を読み込み、時間長 T を [:max_length] で切って返す。
    """
    load_dir = os.path.join(base_dir, str(mouse_id), experience_level, image_set)
    filename = f"spike_data_{mouse_id}_{experience_level}_{image_set}_{trial_type}_{specific_image}.npy"
    filepath = os.path.join(load_dir, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    spike_data = np.load(filepath)[:max_length]
    print(f"Loaded data from: {filepath}")
    return spike_data

def _plot_spike_subdata(
    sub_data, dt, max_time, bins,
    experience_level, image_set, specific_image,
    figure_label="",
    analysis_summary=None,
    ncols=6,
    put_summary_on_last_subplot=False,
    return_metrics=False
):
    """
    sub_data: {mouse_id: {'active': spikes_ac, 'passive': spikes_pa}}
    左: 時間ごとの集団スパイク率（Passiveを背面→Activeを前面に重ねる）
    右: ニューロンごとのスパイク率分布（同じく Passive→Active の順）
    """
    import math
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    # ---- 色と重ね順（Active を前面へ）----
    ACTIVE_COLOR  = 'tab:red'
    PASSIVE_COLOR = 'tab:blue'
    ALPHA_FILL    = 0.55
    Z_BACK, Z_FRONT = 1, 2  # 背面 / 前面の zorder

    num_mice = len(sub_data)
    if num_mice == 0:
        return None

    total_subplots = 2 * num_mice + (1 if (put_summary_on_last_subplot and analysis_summary is not None) else 0)
    nrows = math.ceil(total_subplots / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axs = axs.flatten()

    cv_metrics = {}
    firing_rate_metrics = {}

    for i, (mouse_id, spikes) in enumerate(sub_data.items()):
        idx_left = 2 * i
        idx_right = 2 * i + 1

        spikes_ac = spikes['active']
        spikes_pa = spikes['passive']

        # ===== 左：時間方向（試行×ニューロン平均） =====
        time_bins = np.linspace(0, spikes_ac.shape[0] * dt, spikes_ac.shape[0])
        rate_ac_time = np.mean(spikes_ac, axis=(1, 2))
        rate_pa_time = np.mean(spikes_pa, axis=(1, 2))

        # Passive（背面）→ Active（前面）の順に描画
        axs[idx_left].bar(time_bins, rate_pa_time, width=dt,
                          color=PASSIVE_COLOR, alpha=ALPHA_FILL,
                          label='Passive Replay', zorder=Z_BACK)
        axs[idx_left].bar(time_bins, rate_ac_time, width=dt,
                          color=ACTIVE_COLOR, alpha=ALPHA_FILL,
                          label='Active Behavior', zorder=Z_FRONT)

        axs[idx_left].set_ylabel('Spike probability')
        axs[idx_left].set_xlabel('Time [s]')
        axs[idx_left].set_title(
            f'Mouse {mouse_id} - {experience_level} {image_set} {specific_image}\n'
            f'Spikes: T={spikes_ac.shape[0]}, R={spikes_ac.shape[1]}, N={spikes_ac.shape[2]}'
        )
        axs[idx_left].set_xlim([0, max_time])

        # 凡例は Active→Passive の順で表示（プロキシを使用）
        leg_handles_left = [
            Patch(facecolor=ACTIVE_COLOR,  alpha=ALPHA_FILL, label='Active Behavior'),
            Patch(facecolor=PASSIVE_COLOR, alpha=ALPHA_FILL, label='Passive Replay'),
        ]
        axs[idx_left].legend(handles=leg_handles_left, loc='best')

        # ===== 右：ニューロンごとの平均スパイク率 (/s) =====
        rate_ac_neuron = np.mean(spikes_ac, axis=(0, 1)) / dt
        rate_pa_neuron = np.mean(spikes_pa, axis=(0, 1)) / dt

        # 共通ビン
        bins_common = np.histogram_bin_edges(
            np.concatenate([rate_ac_neuron, rate_pa_neuron]), bins='fd'
        )

        def coefficient_of_variation(x):
            return np.std(x) / np.mean(x)

        cv_ac = float(coefficient_of_variation(rate_ac_neuron))
        cv_pa = float(coefficient_of_variation(rate_pa_neuron))
        cv_metrics[mouse_id] = {"cv_ac": cv_ac, "cv_pa": cv_pa}

        # 発火確率の全体平均（スカラー）
        firing_rate_metrics[mouse_id] = {
            "mean_rate_ac": float(np.mean(spikes_ac)),
            "mean_rate_pa": float(np.mean(spikes_pa)),
        }

        # Passive（背面）→ Active（前面）の順にヒストを描画
        axs[idx_right].hist(rate_pa_neuron, bins=bins_common,
                            color=PASSIVE_COLOR, alpha=ALPHA_FILL,
                            label=f'Passive (CV={cv_pa:.2f})', zorder=Z_BACK)
        axs[idx_right].hist(rate_ac_neuron, bins=bins_common,
                            color=ACTIVE_COLOR, alpha=ALPHA_FILL,
                            label=f'Active (CV={cv_ac:.2f})', zorder=Z_FRONT)

        axs[idx_right].set_ylabel('Neuron count')
        axs[idx_right].set_xlabel('Spike rate (/s)')
        axs[idx_right].set_xlim([0, 45])
        axs[idx_right].set_title(f'Mouse {mouse_id} - Spike rate distribution')

        # 凡例は Active→Passive の順で表示（プロキシ）
        leg_handles_right = [
            Patch(facecolor=ACTIVE_COLOR,  alpha=ALPHA_FILL, label=f'Active (CV={cv_ac:.2f})'),
            Patch(facecolor=PASSIVE_COLOR, alpha=ALPHA_FILL, label=f'Passive (CV={cv_pa:.2f})'),
        ]
        axs[idx_right].legend(handles=leg_handles_right, loc='best')

    # ---- サマリー欄（任意）----
    if put_summary_on_last_subplot and (analysis_summary is not None):
        idx = 2 * num_mice
        axs[idx].axis('off')
        text = (
            f"R: Mean={analysis_summary['R']['mean']:.2f}, Std={analysis_summary['R']['std']:.2f}, "
            f"Min={analysis_summary['R']['min']}, Max={analysis_summary['R']['max']}\n"
            f"  Min Mouse={analysis_summary['R']['min_mouse_id']}, Max Mouse={analysis_summary['R']['max_mouse_id']}\n"
            f"N: Mean={analysis_summary['N']['mean']:.2f}, Std={analysis_summary['N']['std']:.2f}, "
            f"Min={analysis_summary['N']['min']}, Max={analysis_summary['N']['max']}\n"
            f"  Min Mouse={analysis_summary['N']['min_mouse_id']}, Max Mouse={analysis_summary['N']['max_mouse_id']}"
        )
        axs[idx].text(0.5, 0.5, text, fontsize=12, ha='center', va='center', wrap=True)

    # 余白面を消す
    for k in range(total_subplots, len(axs)):
        axs[k].axis('off')

    fig.suptitle(f"{figure_label}\n{experience_level} {image_set} {specific_image}", fontsize=16)
    plt.tight_layout()

    return (fig, cv_metrics, firing_rate_metrics) if return_metrics else fig


# =========================================================
# 4) 図の保存＋ JSON メトリクス保存（出力ルートを outdir_root に統一）
# =========================================================
def show_combined_population_rate(
    spikes_data, dt, max_time, bins='auto',
    experience_level="", image_set="", specific_image="",
    analysis_summary=None, ncols=6,
    outdir_root=None, max_length=75
):
    """
    1〜18匹を1枚目、19匹以降＋サマリーを2枚目に描画。
    図と JSON（CV と発火“平均”）を outdir_root/{exp}/{set}/{image}/ に保存。
    〈PDFファイル名〉
      - Fig. S1. Spike-rate dynamics and distributions for mice 1-18.pdf
      - Fig. S2. Spike-rate dynamics and distributions for mice 19-37.pdf
    """
    if outdir_root is None:
        outdir_root = os.path.join('fig', 'im03', f'maxlen{max_length}', 'rat_comparison')
    base_save_dir = os.path.join(outdir_root, experience_level, image_set, specific_image)
    os.makedirs(base_save_dir, exist_ok=True)

    # 1〜18 と 19以降に分割
    all_ids = list(spikes_data.keys())
    sub_data_1 = {m: spikes_data[m] for i, m in enumerate(all_ids) if i < 18}
    sub_data_2 = {m: spikes_data[m] for i, m in enumerate(all_ids) if i >= 18}

    # --- 図1（mice 1-18）---
    res1 = _plot_spike_subdata(
        sub_data=sub_data_1, dt=dt, max_time=max_time, bins=bins,
        experience_level=experience_level, image_set=image_set, specific_image=specific_image,
        figure_label="(1 to 18 mice)", analysis_summary=None, ncols=ncols,
        put_summary_on_last_subplot=False, return_metrics=True
    )
    fig1_path = None
    cv_metrics1, fr_metrics1 = {}, {}
    if res1 is not None:
        fig1, cv_metrics1, fr_metrics1 = res1
        # ★ ここでファイル名を指定どおりに固定
        fig1_filename = "Fig. S1. Spike-rate dynamics and distributions for mice 1-18.pdf"
        fig1_path = os.path.join(base_save_dir, fig1_filename)
        fig1.savefig(fig1_path, bbox_inches="tight")
        plt.close(fig1)
        print(f"[figure] saved: {fig1_path}")

    # --- 図2（mice 19-37 + summary）---
    res2 = _plot_spike_subdata(
        sub_data=sub_data_2, dt=dt, max_time=max_time, bins=bins,
        experience_level=experience_level, image_set=image_set, specific_image=specific_image,
        figure_label="(19 mice and after + Summary)", analysis_summary=analysis_summary, ncols=ncols,
        put_summary_on_last_subplot=True, return_metrics=True
    )
    fig2_path = None
    cv_metrics2, fr_metrics2 = {}, {}
    if res2 is not None:
        fig2, cv_metrics2, fr_metrics2 = res2
        # ★ ここでファイル名を指定どおりに固定
        fig2_filename = "Fig. S2. Spike-rate dynamics and distributions for mice 19-37.pdf"
        fig2_path = os.path.join(base_save_dir, fig2_filename)
        fig2.savefig(fig2_path, bbox_inches="tight")
        plt.close(fig2)
        print(f"[figure] saved: {fig2_path}")

    # --- JSON 保存（CV と 発火平均） ---
    cv_metrics = {}
    cv_metrics.update(cv_metrics1)
    cv_metrics.update(cv_metrics2)

    fr_metrics = {}
    fr_metrics.update(fr_metrics1)
    fr_metrics.update(fr_metrics2)

    metrics_dir = os.path.join(base_save_dir, "CV_metrics")
    os.makedirs(metrics_dir, exist_ok=True)

    cv_json = os.path.join(metrics_dir, f"CV_{experience_level}_Set_{image_set}_Image_{specific_image}.json")
    with open(cv_json, "w") as f:
        json.dump(cv_metrics, f, indent=4)
    print(f"[json] saved: {cv_json}")

    fr_json = os.path.join(metrics_dir, f"FiringRate_{experience_level}_Set_{image_set}_Image_{specific_image}.json")
    with open(fr_json, "w") as f:
        json.dump(fr_metrics, f, indent=4)
    print(f"[json] saved: {fr_json}")

    return {"figure_part1": fig1_path, "figure_part2": fig2_path, "cv_json": cv_json, "fr_json": fr_json}


# =========================================================
# 5) メイン実行例（outdir_root を 'fig/im03/maxlen75/rat_comparison' に固定）
# =========================================================
# 実行ディレクトリが /ishihara_kine/Non_equ/main_kinetic であることを想定
base_dir = "./spike_datas_all"

mouse_ids = [
    574078, 536480, 532246, 570299, 524925, 599294, 574081, 560356,
    554013, 560771, 541234, 577287, 553960, 533537, 544838, 509808,
    521466, 530862, 574082, 563497, 558306, 533539, 527749, 544836,
    563323, 568963, 548720, 560770, 555304, 553253, 570301, 567286,
    578257, 572846, 524761, 556014, 548721
]

max_length = 75
dt = 0.01
max_time = 0.75
outdir_root = os.path.join('fig', 'im03', f'maxlen{max_length}', 'rat_comparison')

experience_levels_images = {
    'Familiar': {
        'G': ['im036_r', 'im012_r', 'im115_r']
    }
}

for experience_level, images_by_set in experience_levels_images.items():
    for image_set, specific_images in images_by_set.items():
        for specific_image in specific_images:
            print(f"\n=== {experience_level} / {image_set} / {specific_image} ===")

            # すべてのマウスを収集
            combined_data = {}
            for mouse_id in mouse_ids:
                md = process_mouse_data(base_dir, mouse_id, experience_level, image_set, specific_image, max_length)
                if md is not None:
                    combined_data[mouse_id] = md

            if not combined_data:
                print("[skip] no valid data.")
                continue

            # 形状サマリーを保存
            analysis_summary = analyze_and_save_spike_shapes(
                combined_data=combined_data,
                outdir_root=outdir_root,
                experience_level=experience_level,
                image_set=image_set,
                specific_image=specific_image
            )
            print(f"[summary] R/N: {analysis_summary}")

            # 図と JSON を保存
            show_combined_population_rate(
                combined_data, dt, max_time, bins='auto',
                experience_level=experience_level, image_set=image_set, specific_image=specific_image,
                analysis_summary=analysis_summary, ncols=6,
                outdir_root=outdir_root, max_length=max_length
            )



=== Familiar / G / im036_r ===
Loaded data from: ./spike_datas_all/574078/Familiar/G/spike_data_574078_Familiar_G_active_trials_im036_r.npy
Loaded data from: ./spike_datas_all/574078/Familiar/G/spike_data_574078_Familiar_G_passive_trials_im036_r.npy
Loaded data from: ./spike_datas_all/536480/Familiar/G/spike_data_536480_Familiar_G_active_trials_im036_r.npy
Loaded data from: ./spike_datas_all/536480/Familiar/G/spike_data_536480_Familiar_G_passive_trials_im036_r.npy
Loaded data from: ./spike_datas_all/532246/Familiar/G/spike_data_532246_Familiar_G_active_trials_im036_r.npy
Loaded data from: ./spike_datas_all/532246/Familiar/G/spike_data_532246_Familiar_G_passive_trials_im036_r.npy
Loaded data from: ./spike_datas_all/570299/Familiar/G/spike_data_570299_Familiar_G_active_trials_im036_r.npy
Loaded data from: ./spike_datas_all/570299/Familiar/G/spike_data_570299_Familiar_G_passive_trials_im036_r.npy
Loaded data from: ./spike_datas_all/524925/Familiar/G/spike_data_524925_Familiar_G_active_tr